In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from corvia.framework.builder import NetworkBuilder
from corvia.network_states.flow import FlowStore
from corvia.reconstruction.flow_link_balancer import FlowLinkBalancer
from corvia.engine.flow_resolver import FlowResolver
from corvia.engine.validators import ZScoreRejector, RelativeMarginAcceptor
from corvia.loaders.flow_data_loader import VCFlowDataLoader
from corvia.loaders.network.masternet_network import MasterNetLoader

cwd = Path.cwd()

In [ ]:
prefix = f'{(cwd / "data" / "masternet_2026.09.14" / "Masternet").as_posix()}'

# 1. Load the shapefiles into builder-ready records
loader = MasterNetLoader(prefix, dropped_report_path="output_masternet/masternet_dropped_objects.csv")
links_data, nodes_data, counts_data, crs = loader.load()

In [ ]:
# Build the network (nodes/links/sensors -> sections -> GeoDataFrames)
network = NetworkBuilder.from_data("Masternet", links_data, nodes_data, counts_data, crs=crs)
links_gdf, sensors_gdf = network.links_gdf, network.sensors_gdf
print(network.summary())

In [ ]:
# Sensor -> link / section table for further computations
rows = [
    {"code": s.location_id, "link_id": link.link_id, "section_id": link.parent_section.section_id}
    for link in network.links.values()
    for s in link.sensor_locations
]
sensor_table = sensors_gdf.join(pd.DataFrame(rows).set_index("code"), how="left")
print(sensor_table.head(3))
sensor_table.to_csv("output_masternet/masternet_sensor_table.csv", index=True)

In [ ]:
csv_filenames = ["data/Masternet_counts-VC_280926-021026.csv"]

flow_loader = VCFlowDataLoader(
    csv_filenames=csv_filenames,
    agg_freq="1D",
    pae_factor=2.0,
    start="2026-09-29",
    end="2026-09-30",
    vehicle_types=["TOTAL"],
    volume_err_fn=lambda df: df["volume"] * 0.005,
)

day_df = flow_loader.prepare(network, report="output_masternet/masternet_sensor_report.csv")

conditions = [
    (day_df["pct_vol_reconstructed"] > 10.0) | (day_df["pct_vol_reco_typed"] > 5.0),
    (day_df["pct_vol_reconstructed"] >= 3.0)
    & (day_df["pct_vol_reconstructed"] <= 10.0)
    & (day_df["pct_vol_reco_typed"] <= 5.0),
]
choices = ["held", "disputed"]
day_df["screening"] = np.select(conditions, choices, default="accepted")

print("Screening-overzicht:")
print(day_df.drop_duplicates(subset=["source_id"])["screening"].value_counts())

store = FlowStore.from_dataframe(day_df)

In [ ]:
report_df = pd.read_csv("output_masternet/masternet_sensor_report.csv", delimiter=";")
csv_sensors = set(report_df["LOCPOST"].astype(int).astype(str))

network_sensors = set()
for link in network.links.values():
    for sensor in link.sensor_locations:
        network_sensors.add(sensor.location_id)

matching = csv_sensors & network_sensors
only_in_csv = csv_sensors - network_sensors
only_in_network = network_sensors - csv_sensors

print(f"Overeenkomende sensoren:      {len(matching)}")
print(f"Enkel in tellingenbestand:    {len(only_in_csv)}")
print(f"Enkel in netwerk:             {len(only_in_network)}")

In [ ]:
vehicle_type = "TOTAL"

reconstructors = [
    FlowLinkBalancer(direction="upstream", min_obs_degree=1, max_link_degree=3,
                      weight=1, allow_partial_fallback=True),
    FlowLinkBalancer(direction="upstream", min_obs_degree=2, max_link_degree=4,
                      weight=1, allow_partial_fallback=True),
    FlowLinkBalancer(direction="downstream", min_obs_degree=1, max_link_degree=3,
                      weight=1, allow_partial_fallback=True),
    FlowLinkBalancer(direction="downstream", min_obs_degree=2, max_link_degree=4,
                      weight=1, allow_partial_fallback=True),
]

resolver = FlowResolver(
    reconstructors=reconstructors,
    rejector=ZScoreRejector(use_errors=True, significance_level=0.10, median_mixing_fraction=0.),
    acceptor=None, #RelativeMarginAcceptor(significance_level=0.05, max_percent_deviation=2.0, noise_floor=10., use_errors=True),
    max_iterations=50,
    name="voorbeeld-netwerk",
    greedy_elimination=True,
    greedy_adaptive_batch=True,
    greedy_k_max=10,
)

resolved_store = resolver.run(
    store=store,
    network=network,
    periods=[pd.Timestamp("2026-09-29")],
    vehicle_types=[vehicle_type],
)

In [ ]:
import pickle

with open("output_masternet/masternet_resolved_flowstore.pkl", "wb") as f:
    pickle.dump(resolved_store, f)

In [ ]:
def get_sensor_vs_resolved(flow_df: pd.DataFrame) -> pd.DataFrame:
    """
    For every observation, pairs its own reading with the resolved consensus
    for its (road_section_id, timestamp, vehicle_type), and computes their
    relative difference with propagated error.

    Returns
    -------
    pd.DataFrame, indexed like the store's observation rows, with columns:
        source_id, road_section_id, timestamp, vehicle_type,
        volume, volume_err, screening, validation,
        resolved_volume, resolved_volume_err,
        relative_diff, relative_diff_err
    """
    df = flow_df.copy()

    obs_cols = ["source_id", "road_section_id", "timestamp", "vehicle_type",
                "volume", "volume_err", "screening", "validation"]
    obs_rows = df.loc[df["source_type"] == FlowStore.SOURCE_OBS, obs_cols].reset_index()

    resolved_rows = (
        df.loc[df["source_type"] == FlowStore.SOURCE_RES,
               ["road_section_id", "timestamp", "vehicle_type", "volume", "volume_err"]]
        .rename(columns={"volume": "resolved_volume", "volume_err": "resolved_volume_err"})
    )

    dupes = resolved_rows.duplicated(subset=["road_section_id", "timestamp", "vehicle_type"])
    if dupes.any():
        raise ValueError(f"Expected one resolved row per coordinate, found {int(dupes.sum())} duplicates.")

    merged = (
        obs_rows.merge(resolved_rows, on=["road_section_id", "timestamp", "vehicle_type"], how="left")
        .set_index("index")
    )
    merged.index.name = None

    merged["relative_diff"] = (merged["volume"] - merged["resolved_volume"]) / merged["resolved_volume"]
    merged["relative_diff_err"] = (
        np.sqrt(merged["volume_err"] ** 2 + merged["resolved_volume_err"] ** 2)
        / merged["resolved_volume"].abs()
    )

    merged["source_id"] = merged["source_id"].str.replace("sensor:", "", regex=True).astype(int)

    return merged


report = get_sensor_vs_resolved(resolved_store.dataframe)
report.to_csv(f"output_masternet/reconstruction_report_{vehicle_type}.csv", index=False)
report.head()


In [ ]:
report["source_id"] = report["source_id"].astype(str)
sensors_out = sensor_table.merge(report, left_index=True, right_on="source_id", how="left")
sensors_out.to_file("output_masternet/masternet_sensor_resolved.gpkg", layer="locposten_TOTAL_resolved", driver="GPKG")